# Prática Complementar: aprendendo a lógica XOR com uma rede neural

O XOR é o exemplo clássico de problema **não linearmente separável**.
Vamos atacá-lo por três caminhos: aumentar o modelo, normalizar os dados
e, por fim, criar uma feature de interação.

**PS:** *Esta prática é um material complementar para estudo individual. Para entrar em detalhes, realizaremos uma outra prática acompanhada na aula 2.*

## [1] Importação de bibliotecas

In [ ]:
"""
Define uma seed global para garantir reprodutibilidade máxima
no treinamento com TensorFlow/Keras.

NOTE 1: Para aulas, deixo essa parte comentada para demonstrar o
        comportamento estocástico das práticas com DL.
NOTE 2: Na prática diária com DL, determinamos as SEEDs somente no 
        split dos dados, pois operações em GPU sofrem perda de 
        performance quando forçadas ao determinismo.
NOTE 3: Em alguns casos de treinamento avançado com DL, não é 
        possível estabelecer 100% de determinismo devido à alta
        dependência de operações paralelizadas.
"""
# 1. Variáveis de ambiente (precisam ser definidas ANTES de importar TF!
import os
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'      # força kernels determinísticos
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'    # determinismo em GPU (cuDNN)


import math
import tensorflow as tf
import keras
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from absl import logging

# seeds determinadas após importação dos módulos
keras.utils.set_random_seed(SEED)
# 3. Habilita operações determinísticas no TF (TF >= 2.8)
# NOTE: Haverá perda de performance
# NOTE: Em alguns casos, ainda não será 100% determinístico (paralelismo!)
tf.config.experimental.enable_op_determinism()

# silenciar alguns avisos do tf que não serão relevantes
tf.get_logger().setLevel('ERROR')
logging.set_verbosity(logging.ERROR)

## [2] Modelos usados nesta prática

Três arquiteturas de MLP, da mais simples à mais profunda. Execute esta
célula uma vez; depois basta chamar `mlp_1(...)`, `mlp_2(...)` ou `mlp_3(...)`.

In [ ]:
def mlp_1(input_shape):
    model = keras.Sequential([
        keras.layers.Input(input_shape, name='input'),        
        keras.layers.Dense(2, activation='relu', name='hidden_1'),               
        keras.layers.Dense(1, activation='sigmoid', name='output')
    ], name='MLP1')

    return model


def mlp_2(input_shape):
    model = keras.Sequential([
        keras.layers.Input(input_shape, name='input'),        
        keras.layers.Dense(64, activation='relu', name='hidden_1'),                
        keras.layers.Dense(64, activation='relu', name='hidden_2'),                
        keras.layers.Dense(1, activation='sigmoid', name='output')
    ], name='MLP2')

    return model


def mlp_3(input_shape):
    model = keras.Sequential([
        keras.layers.Input(input_shape, name='input'),        
        keras.layers.Dense(4, activation='relu', name='hidden_1'),
        keras.layers.Dense(4, activation='relu', name='hidden_2'),
        keras.layers.Dense(4, activation='relu', name='hidden_3'),
        keras.layers.Dense(1, activation='sigmoid', name='output')
    ], name='MLP3')

    return model

## [3] Funções utilitárias

As células desta seção apenas **definem** as funções usadas no restante do
notebook. Execute todas uma vez, no início da sessão.

### `smooth_dataframe` - suavização das séries do histórico

*não se preocupe com os detalhes dessa função, é apenas utilitária*

In [ ]:
def smooth_dataframe(df, window_size=10):
    smoothed_data = {
        col: np.convolve(df[col], np.ones(window_size) / window_size, mode='valid')
        for col in df.columns
    }
    return pd.DataFrame(smoothed_data)

### `scatter_binary` - comparação entre previsões e valores reais

*não se preocupe com os detalhes dessa função, é apenas utilitária*

In [ ]:
def scatter_binary(x, y_true, y_pred):
    """
    Gera um gráfico de dispersão para comparar as previsões binárias com os valores reais.
    Parâmetros:
    - x (numpy.ndarray): Dados de entrada com duas dimensões.
    - y_true (numpy.ndarray): Valores reais (0 ou 1).
    - y_pred (numpy.ndarray): Valores previstos (0 ou 1).
    """
    # y_pred precisa estar em formato vetorial ~ (N,)
    preds = np.reshape(y_pred, (-1,))
    
    # scatter de y_pred
    plt.figure(figsize=(10, 6))
    plt.scatter(x[preds<=0.5, 0], x[preds<=0.5, 1], s=50, marker='o', c='blue', label='pred 0', alpha=0.5)
    plt.scatter(x[preds>0.5, 0], x[preds>0.5, 1], s=50, marker='s', c='red', label='pred 1', alpha=0.5)

    # scatter de y_true
    plt.scatter(x[y_true==0, 0], x[y_true==0, 1], s=50, marker='s', c='blue', label='true 0', alpha=0.5)
    plt.scatter(x[y_true==1, 0], x[y_true==1, 1], s=50, marker='^', c='red', label='true 1', alpha=0.5)
    plt.title('True vs Preds')
    plt.xlabel('X1')
    plt.ylabel('X2')
    plt.legend()
    plt.show()

### `plot_history` - curvas de loss e accuracy ao longo das épocas

*não se preocupe com os detalhes dessa função, é apenas utilitária*

In [ ]:
def plot_history(history_df, skip_epochs=0, smooth=1):
    """
    Plots training and (optional) validation metrics from a Keras history DataFrame.

    Args:
        history_df (pd.DataFrame): DataFrame containing Keras training history.
                                   Expected to have columns like 'loss', 'accuracy', etc.
                                   Validation columns (if present) must be named 'val_<metric>'.
        skip_epochs (int): Number of initial rows (epochs) to skip before plotting.
        smooth (int): Window length for smoothing. If > 1, the DataFrame will be smoothed first.
                      (Assumes you have a function `smooth_dataframe(df, window)` defined elsewhere.)
    """
    if not isinstance(history_df, pd.DataFrame):
        raise TypeError("history_df must be a pandas DataFrame.")
    if history_df.empty:
        print("History DataFrame is empty. Nothing to plot.")
        return

    # Optionally skip some initial epochs
    history_df = history_df.iloc[skip_epochs:].copy()

    # If the user requested smoothing, apply it here.
    if smooth > 1:
        history_df = smooth_dataframe(history_df, smooth)

    # Identify all "training" columns (i.e. those that do NOT start with 'val_').
    # For each such column, check whether a corresponding 'val_<col>' exists.
    base_metric_names = []
    metric_has_val = {}  # dict: metric_name -> True/False
    for col in history_df.columns:
        if not col.startswith('val_'):
            base_metric_names.append(col)
            if f"val_{col}" in history_df.columns:
                metric_has_val[col] = True
            else:
                metric_has_val[col] = False
                print(f"Warning: Found training metric '{col}' but no 'val_{col}'. Will plot training-only.")

    if not base_metric_names:
        print("No training metrics found to plot.")
        return

    num_metrics = len(base_metric_names)

    # Arrange subplots: 2 columns, nrows = ceil(num_metrics/2)
    ncols = 2
    nrows = math.ceil(num_metrics / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 6, nrows * 5))

    # Flatten axes into a 1-D list for easy indexing
    if num_metrics == 1:
        axes_flat = [axes]  # single Axes object
    elif nrows == 1 or ncols == 1:
        axes_flat = axes  # 1D array of Axes
    else:
        axes_flat = axes.flatten()

    for i, metric_name in enumerate(base_metric_names):
        ax = axes_flat[i]
        train_col = metric_name
        val_col = f"val_{metric_name}"
        
        # Plot training curve
        ax.plot(
            history_df[train_col],
            label=f"Training {metric_name.replace('_', ' ').title()}",
            # marker="o",
            linestyle="-"
        )

        # If a validation column exists, plot it too
        if metric_has_val[metric_name]:
            ax.plot(
                history_df[val_col],
                label=f"Validation {metric_name.replace('_', ' ').title()}",
                # marker="x",
                linestyle="--"
            )

        ax.set_title(f"{metric_name.replace('_', ' ').title()} Over Epochs")
        ax.set_xlabel("Epoch")
        ax.set_ylabel(metric_name.replace("_", " ").title())
        ax.legend()
        ax.grid(True, linestyle="--", alpha=0.7)

        # Compute y-limits using whichever columns are available
        train_min = history_df[train_col].min()
        train_max = history_df[train_col].max()
        if metric_has_val[metric_name]:
            val_min = history_df[val_col].min()
            val_max = history_df[val_col].max()
            overall_min = min(train_min, val_min)
            overall_max = max(train_max, val_max)
        else:
            overall_min = train_min
            overall_max = train_max

        if pd.notna(overall_min) and pd.notna(overall_max) and (overall_max > overall_min):
            padding = (overall_max - overall_min) * 0.1
            ax.set_ylim(overall_min - padding, overall_max + padding)

    # If there are any unused subplot slots, delete them
    for j in range(num_metrics, nrows * ncols):
        fig.delaxes(axes_flat[j])

    plt.tight_layout()
    plt.show()

### `make_xor_data` - geração dos dados XOR

A lógica XOR possui 4 regiões, cada uma representada aqui por uma
gaussiana 2D centrada em um dos quadrantes. O rótulo (target) segue a
regra do "ou exclusivo": positivo apenas quando os sinais dos dois
centros divergem (quadrantes opostos).

In [ ]:
def make_xor_data(n=200, mean=10, std=4, offset=10):
    """
    Gera um dataset sintético que representa a lógica XOR.

    Parâmetros
    ----------
    n : int
        Número total de amostras (dividido entre as 4 regiões).
    mean : float
        Distância dos centros das gaussianas em relação à origem.
    std : float
        Desvio padrão (dispersão) para simular ruído nos dados.
    offset : float
        Deslocamento aplicado aos dados para criar assimetria de escala.

    Retorna
    -------
    X : np.ndarray, shape (n, 2)
        Coordenadas dos pontos.
    y : np.ndarray, shape (n,)
        Rótulos binários (1 = quadrantes opostos, 0 = mesmo quadrante).
    """
    centers = [[mean, mean], [-mean, -mean], [mean, -mean], [-mean, mean]]

    data, targets = [], []
    for center in centers:
        points = np.random.normal(loc=center, scale=std, size=(n // 4, 2))
        label = 1 if np.mean(center) == 0 else 0  # lógica XOR
        data.append(points + offset)
        targets.append(np.full(n // 4, label))

    X = np.concatenate(data, axis=0)
    y = np.concatenate(targets, axis=0)
    return X, y

### `plot_scatter_xor` - visualização em scatter

In [ ]:
def plot_scatter_xor(X, y, title='Lógica XOR'):
    """Exibe um scatter 2D colorido pelo rótulo binário y."""
    plt.figure(figsize=(10, 6))
    plt.scatter(X[y == 0, 0], X[y == 0, 1], s=50, marker='o', c='blue', label='0')
    plt.scatter(X[y == 1, 0], X[y == 1, 1], s=50, marker='d', c='red', label='1')
    plt.title(title)
    plt.xlabel('A')
    plt.ylabel('B')
    plt.legend()
    plt.show()

### `describe_model` - descrição do modelo

In [ ]:
def describe_model(model):
    model.summary()
    return keras.utils.plot_model(model, show_shapes=True, show_layer_names=True)

### `train_model` - treino + visualização do histórico

In [ ]:
def train_model(model, X, y, epochs=100, batch_size=16, lr=1e-1, verbose=1):
    """
    Compila e treina um modelo, plotando loss e accuracy ao final.

    Retorna
    -------
    pd.DataFrame
        Histórico de treino (uma coluna por métrica).
    """
    model.compile(
        optimizer=keras.optimizers.SGD(learning_rate=lr),
        loss=keras.losses.MeanAbsoluteError(),
        metrics=['accuracy']
    )
    hist = model.fit(
        X, y,
        epochs=epochs,
        batch_size=batch_size,
        shuffle=True,
        verbose=verbose
        )

    history_df = pd.DataFrame(hist.history)
    plot_history(history_df)
    return history_df

## [4] Geração e visualização dos dados

In [ ]:
X, y = make_xor_data(n=200, mean=10, std=3.5, offset=10)
plot_scatter_xor(X, y)

## [5] Tentativa 1: modelo simples (MLP1) sobre os dados crus

Hipótese: um modelo pequeno consegue separar as 4 regiões?

In [ ]:
model = mlp_1(input_shape=X.shape[1:])
describe_model(model)

In [ ]:
train_model(model, X, y)

y_pred = model.predict(X)
scatter_binary(X, y, y_pred)

## [6] Tentativa 2: modelo maior (MLP2) sobre os dados crus

Hipótese: mais neurônios resolvem o problema?

In [ ]:
model = mlp_2(input_shape=X.shape[1:])
describe_model(model)

In [ ]:
train_model(model, X, y)

y_pred = model.predict(X)
scatter_binary(X, y, y_pred)

## [7] Padronização dos dados (0 a 1) + modelo MLP2

Redes neurais treinam melhor quando as entradas estão em escalas
comparáveis. Reescalamos X para o intervalo [0, 1] com `MinMaxScaler`.

In [ ]:
scaler = MinMaxScaler((0, 1))
X_scaled = scaler.fit_transform(X)

print(f'Antes  -> min: {X.min():.2f}, max: {X.max():.2f}')
print(f'Depois -> min: {X_scaled.min():.2f}, max: {X_scaled.max():.2f}')

print('\n Training...')
model = mlp_2(input_shape=X_scaled.shape[1:])
train_model(model, X_scaled, y, verbose=0)

y_pred = model.predict(X_scaled)
scatter_binary(X_scaled, y, y_pred)

## [8] Padronização dos dados (-1 a 1) + modelo MLP2

Testamos agora um intervalo simétrico em torno de 0.

In [ ]:
scaler = MinMaxScaler((-1, 1))
X_scaled = scaler.fit_transform(X)

print(f'Antes  -> min: {X.min():.2f}, max: {X.max():.2f}')
print(f'Depois -> min: {X_scaled.min():.2f}, max: {X_scaled.max():.2f}')

plot_scatter_xor(X_scaled, y, title='Lógica XOR (normalizada)')

In [ ]:
model = mlp_2(input_shape=X_scaled.shape[1:])
describe_model(model)

In [ ]:
train_model(model, X_scaled, y, verbose=0)

y_pred = model.predict(X_scaled)
scatter_binary(X_scaled, y, y_pred,)

## [9] Engenharia de features: Xe = A * B

XOR não é linearmente separável no espaço original (A, B). Mas se
criarmos a feature de interação A*B, os sinais dos quadrantes opostos
se tornam diretamente separáveis por um modelo muito mais simples.

In [ ]:
scaler = MinMaxScaler((-1, 1))
X_scaled = scaler.fit_transform(X)

X_engineered = (X_scaled[:, 0] * X_scaled[:, 1]).reshape(-1, 1)

# NOTE: Voltamos a testar o modelo mais simples!!
model = mlp_1(input_shape=X_engineered.shape[1:])
describe_model(model)

In [ ]:
train_model(model, X_engineered, y, verbose=0)

y_pred = model.predict(X_engineered)
scatter_binary(X_scaled, y, y_pred,)

model.evaluate(X_engineered, y)

## [10] Conclusões

1. Mesmo adicionando ruído e assimetria, a rede neural `mlp_2` conseguiu resolver o XOR
2. Padronizar a escala das features possibilitou diminuir o tamanho do modelo consideravelmente (`mlp_3`)
3. Apesar da rede neural se mostrar eficaz SEM feature eng., observamos que ainda é possível se beneficiar desta técnica (boa para quando conhecemos bem a natureza dos dados): `mlp_1`
